# BRFSS 2025 — Initial Data Exploration

Purpose: verify the official CDC archive, load the SAS Transport dataset, inspect shape/columns, and examine the raw `DIABETE4` target before any cleaning or modeling.

**No model training and no target recoding are performed in this notebook.**


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib
import json
import pandas as pd
from IPython.display import display

EXPECTED_ZIP_SHA256 = "d26a8b391aa2966b58dc2a3ffddf8da050e9f38eb143d0f5ca7b3d21ece08f98"
TARGET = "DIABETE4"

zip_candidates = list(Path("/kaggle/input").rglob("LLCP2025XPT.zip"))
if len(zip_candidates) != 1:
    raise RuntimeError(f"Expected exactly one LLCP2025XPT.zip in /kaggle/input, found {len(zip_candidates)}")

zip_path = zip_candidates[0]
sha256 = hashlib.sha256()
with zip_path.open("rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        sha256.update(chunk)
actual_sha256 = sha256.hexdigest()

print("Archive:", zip_path)
print("SHA-256:", actual_sha256)
if actual_sha256 != EXPECTED_ZIP_SHA256:
    raise RuntimeError("Archive checksum does not match the verified CDC source.")


In [ ]:
working_dir = Path("/kaggle/working/brfss_2025")
working_dir.mkdir(parents=True, exist_ok=True)

with ZipFile(zip_path) as zf:
    members = [info for info in zf.infolist() if not info.is_dir()]
    print("Archive members:")
    for info in members:
        print(f"  {info.filename!r} -> {info.file_size:,} bytes")

    if len(members) != 1:
        raise RuntimeError(f"Expected one data payload in CDC ZIP, found {len(members)}.")

    member = members[0]
    normalized_name = Path(member.filename.strip()).name
    if not normalized_name.lower().endswith(".xpt"):
        raise RuntimeError(f"Unexpected payload filename after normalization: {normalized_name!r}")

    xpt_path = working_dir / normalized_name
    with zf.open(member) as src, xpt_path.open("wb") as dst:
        for chunk in iter(lambda: src.read(8 * 1024 * 1024), b""):
            dst.write(chunk)

print("Extracted:", xpt_path)
print("Extracted size:", f"{xpt_path.stat().st_size:,} bytes")


In [ ]:
df = pd.read_sas(xpt_path, format="xport", encoding="utf-8")

print("Shape:", df.shape)
print("Rows:", f"{len(df):,}")
print("Columns:", df.shape[1])
print("Memory usage:", f"{df.memory_usage(deep=True).sum() / (1024**3):.2f} GiB")

if TARGET not in df.columns:
    raise RuntimeError(f"Target {TARGET!r} not found in dataset.")

display(df.head())


In [ ]:
target_counts = (
    df[TARGET]
    .value_counts(dropna=False)
    .rename_axis(TARGET)
    .reset_index(name="count")
)
target_counts["percent"] = target_counts["count"] / len(df) * 100

print("Raw DIABETE4 distribution (no recoding yet):")
display(target_counts)
target_counts.to_csv("/kaggle/working/target_distribution.csv", index=False)


In [ ]:
column_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(dtype) for dtype in df.dtypes],
    "missing_count": df.isna().sum().values,
    "missing_pct": (df.isna().mean().values * 100),
    "n_unique": [df[col].nunique(dropna=True) for col in df.columns],
}).sort_values(["missing_pct", "column"], ascending=[False, True])

print("Columns with the highest missingness:")
display(column_summary.head(30))

print("All column names:")
print(df.columns.tolist())

column_summary.to_csv("/kaggle/working/column_summary.csv", index=False)


In [ ]:
overview = {
    "source_archive": zip_path.name,
    "archive_sha256": actual_sha256,
    "rows": int(df.shape[0]),
    "columns": int(df.shape[1]),
    "target": TARGET,
    "target_missing": int(df[TARGET].isna().sum()),
    "memory_gib": round(float(df.memory_usage(deep=True).sum() / (1024**3)), 3),
}

with open("/kaggle/working/dataset_overview.json", "w", encoding="utf-8") as f:
    json.dump(overview, f, indent=2)

print(json.dumps(overview, indent=2))
print("Initial exploration complete.")
